# Chicago taxi trips, 11–17 September 2023: weekly summary

*Reported trips only (not total market volume). 135,259 of 135,768 raw trips are analysed; 509 rows (0.4%) were excluded as unusable (see Section 2).*

- **When:** Mon–Thu are almost identical at about 22,000 trips a day (Wed highest at 22,622, a lead too small to matter). Friday is lower (20,151) and the weekend much lower: Saturday is the quietest day (12,643) and Sunday has 13,317. A weekend day averages about 12,980 trips against about 21,860 on a weekday. On weekdays, demand climbs sharply from 06:00 to 09:00 and stays high until about 18:00. The busiest hour is 17:00 (9,468 trips over the week) and the quietest is 03:00 (627). Weekends have no morning peak, build slowly to a plateau around 12:00–19:00, and have a bigger share of after-midnight trips (about 8.5% of trips start 00:00–04:59, against 2.5–5% on weekdays).
- **Where:** Pickups are heavily concentrated. Areas 76, 8 and 32 hold 60.5% of trips (23.3%, 20.9% and 16.3%), and the top 10 areas hold 85.1%. The same 10 areas lead by trip count and by total `trip_total`, but area 76 alone brings in 43% of revenue (\$1.73M of \$4.02M) because its average trip is \$54.74, against about \$18–19 for areas 8 and 32. **2.8% of trips (3,747) have no pickup community area.**
- **Payment and tips:** Credit cards carry 44.5% of trips and 59.6% of revenue (average fare \$28.72). Cash is 29.3% of trips but 19.6% of revenue (average fare \$18.14), and `Mobile` is 13.7% of trips. Where tips are recorded, they are large: 22.2% of fare on credit cards and 20.5% on mobile payments (about 93% of trips in each tipped). Cash shows about 0.1%, which reflects tips not being recorded, not riders not tipping.
- **Companies:** Flash Cab (25,896 trips, 19.1%) and Taxi Affiliation Services (25,748, 19.0%) are effectively tied and together carry 38% of trips. The top 5 carry 71% and the top 10 carry 92%. The same business appears under two spellings in the raw data, and merging them moves Taxicab Insurance Agency from 4th to 3rd (16,172 trips).
- **Data quality:** The cleaning rules affect under 0.4% of rows (they hold 1.4% of raw revenue, \$21.9k of it in three corrupt toll/extras amounts of \$4,100–\$8,900 each). A stricter variant that also drops rows with bad miles/duration data, zero fares and tips above the fare (a further 2.7%) leaves every ranking and conclusion above unchanged; the percentages move by less than 0.5 points (Section 7).

## 1. Setup and loading

**Data:** `trips.csv` (reported Chicago taxi trips, Mon 2023-09-11 to Sun 2023-09-17), columns described in `dataset.md`.
Every figure below describes **reported trips**, not total market volume (`dataset.md`: not all trips are reported).

**Basic assumptions**
- Timestamps are Chicago local time (the file has no time zone) and are rounded to the nearest 15 minutes, so "hour of start" is the hour of the *rounded* timestamp (a trip starting 13:55 shows as 14:00). Day of week is taken from the same rounded timestamp.
- **Revenue** means the sum of `trip_total` (it includes tips, tolls and extras). **Fare** means the `fare` column only.
- Geography uses `pickup_community_area` only (shown as the numeric code; community-area names are outside data and out of scope). Census tracts are not used.

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

# Chart styling: one blue (sequential / single series) and one orange (second category),
# neutral grays for chrome. Text uses neutral ink, never the series colour.
BLUE, ORANGE, GRAY = '#2a78d6', '#eb6834', '#b9b8b1'
INK, INK2, MUTED, GRID, SURFACE = '#0b0b0b', '#52514e', '#898781', '#e1e0d9', '#fcfcfb'
plt.rcParams.update({
    'figure.facecolor': SURFACE, 'axes.facecolor': SURFACE, 'savefig.facecolor': SURFACE,
    'axes.edgecolor': '#c3c2b7', 'axes.labelcolor': INK2, 'axes.titlecolor': INK,
    'axes.titlesize': 13, 'axes.titleweight': 'bold', 'axes.titlelocation': 'left',
    'axes.labelsize': 10.5, 'xtick.color': MUTED, 'ytick.color': MUTED, 'text.color': INK,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': GRID, 'grid.linewidth': 1, 'axes.axisbelow': True,
    'figure.dpi': 100, 'font.family': 'sans-serif',
})
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 30)
pd.options.display.float_format = '{:,.2f}'.format

raw = pd.read_csv('trips.csv')
raw['start'] = pd.to_datetime(raw['trip_start_timestamp'])
print(f"Rows: {len(raw):,}   Columns: {raw.shape[1]}")
print(f"Start timestamps: {raw.start.min()}  ->  {raw.start.max()}")
print("Duplicate trip_id:", raw.trip_id.duplicated().sum(), "  Fully duplicated rows:", raw.duplicated().sum())

## 2. Data quality checks and cleaning

First, what is missing and what looks implausible. Nothing is changed in this step.

In [ ]:
miss = raw.isna().sum()
miss = miss[miss > 0].to_frame('missing'); miss['% of rows'] = miss['missing'] / len(raw) * 100
display(miss)

money = ['fare', 'tips', 'tolls', 'extras', 'trip_total']
print("\nRows missing the money fields:", raw.fare.isna().sum(), "| all five money fields missing together:", raw[money].isna().all(axis=1).sum())
print("Payment type of those rows:", raw[raw.fare.isna()].payment_type.value_counts().to_dict())
display(raw[['trip_seconds', 'trip_miles', 'fare', 'tips', 'tolls', 'extras', 'trip_total']].describe(percentiles=[.5, .99, .999]).T)

**What the checks show** (details in the cells below):
- 324 rows (0.24%) have no money fields at all, almost all `Cash`. Revenue and tips cannot be computed for them.
- A handful of rows carry absurd amounts, e.g. `tolls` or `extras` of about \$4,444 (a keying error).
- A cluster of card trips has a fare of \$195–\$550 with **0 miles and a few seconds** of duration, and sometimes a 20–25% tip on top. A fare like that is not supported by any distance driven.
- `trip_total` does **not** always equal fare + tips + tolls + extras: it is never lower, and is higher by a small amount on many card trips.
- Many rows have zero miles and/or zero seconds, a few have speeds above 100 mph, and some have durations of many hours.

In [ ]:
# --- Rule definitions (all computed on the raw data so each count stands alone) ---
parts = raw[['fare', 'tips', 'tolls', 'extras']].sum(axis=1, min_count=4)
gap = (raw.trip_total - parts).round(2)           # trip_total minus the sum of its parts
mph = raw.trip_miles / (raw.trip_seconds / 3600)

excl = {
 'E1  Money fields missing (fare, tips, tolls, extras, trip_total)': raw.fare.isna(),
 'E2  Corrupt amount: tolls or extras above $1,000':               (raw.tolls > 1000) | (raw.extras > 1000),
 'E3  Fare >= $100 not supported by distance (0 miles, or > $20 per mile)':
        (raw.fare >= 100) & ((raw.trip_miles == 0) | (raw.fare / raw.trip_miles > 20)),
}
flags = {
 'F1  trip_total differs from fare+tips+tolls+extras': gap.abs() > 0.01,
 'F2  ... of which trip_total is more than $1 above the parts': gap > 1,
 'F3  Zero miles': raw.trip_miles == 0,
 'F4  Zero seconds': raw.trip_seconds == 0,
 'F5  Zero miles AND zero seconds': (raw.trip_miles == 0) & (raw.trip_seconds == 0),
 'F6  Miles > 0 but zero seconds': (raw.trip_miles > 0) & (raw.trip_seconds == 0),
 'F7  Speed above 100 mph': mph > 100,
 'F8  Duration over 12 hours': raw.trip_seconds > 12 * 3600,
 'F9  trip_seconds missing': raw.trip_seconds.isna(),
 'F10 Fare is exactly 0': raw.fare == 0,
 'F11 Tip larger than the fare': raw.tips > raw.fare,
 'F12 No pickup community area': raw.pickup_community_area.isna(),
}
print("trip_total minus sum of parts -> min %.2f, max %.2f" % (gap.min(), gap.max()))
print("Gap values:", gap.value_counts().head(6).to_dict())
print("Gap by payment type (rows with gap != 0):", raw[gap.abs() > .01].payment_type.value_counts().to_dict())
print("Total $ of gap: %s" % format(gap.sum(), ',.2f'), "of total trip_total", format(raw.trip_total.sum(), ',.2f'))

In [ ]:
# Exclusions applied in order; 'newly removed' avoids double counting overlaps.
keep = pd.Series(True, index=raw.index)
rows = []
for name, m in excl.items():
    new = (m & keep).sum()
    rows.append({'rule': name, 'rows matching rule': int(m.sum()), 'newly removed': int(new)})
    keep &= ~m
excl_tbl = pd.DataFrame(rows)
excl_tbl.loc[len(excl_tbl)] = ['TOTAL EXCLUDED', np.nan, excl_tbl['newly removed'].sum()]
display(excl_tbl.style.format({'rows matching rule': '{:,.0f}', 'newly removed': '{:,.0f}'}))

df = raw[keep].copy()
print(f"\nAnalysis set: {len(df):,} trips ({len(df)/len(raw):.2%} of the {len(raw):,} raw rows)")
ex = raw[~keep]
print("Revenue (trip_total) in excluded rows with money: $%s of $%s raw (%.1f%%)" % (
      format(ex.trip_total.sum(), ',.0f'), format(raw.trip_total.sum(), ',.0f'), ex.trip_total.sum() / raw.trip_total.sum() * 100))
display(ex[['trip_seconds','trip_miles','fare','tips','tolls','extras','trip_total','payment_type']].sort_values('trip_total', ascending=False).head(8))

In [ ]:
# Flags: these rows are KEPT. Counts are on the raw data and rules overlap, so they do not sum.
f_tbl = pd.DataFrame({'rows flagged (raw)': {k: int(v.sum()) for k, v in flags.items()}})
f_tbl['of which in analysis set'] = [int((v & keep).sum()) for v in flags.values()]
f_tbl['% of analysis set'] = f_tbl['of which in analysis set'] / len(df) * 100
display(f_tbl.style.format({'rows flagged (raw)': '{:,.0f}', 'of which in analysis set': '{:,.0f}', '% of analysis set': '{:.2f}'}))

### Cleaning decisions and assumptions

**One analysis set is used for every question and every table below** (this is what "applied consistently" means here), so trip counts, revenue and shares always refer to the same trips.

**Excluded** (rows removed, counts in the table above):
- **E1 – money fields missing.** Revenue, fare and tips can't be computed, and these are almost all cash trips, so dropping them removes cash trips slightly more than other types (about 0.8% of cash trips).
- **E2 – tolls or extras above \$1,000.** Clear keying errors (≈ \$4,444 each); they alone would add about \$21k to revenue.
- **E3 – fare ≥ \$100 with 0 miles, or more than \$20 per mile.** Genuine long trips such as 195 miles for \$471 pass this test; \$195–\$550 for 0 miles in a few seconds do not. Using trip_total as reported for them would inflate revenue and, with the large tips on some, the tip percentage.

**Kept but flagged** (the table above):
- **Zero-mile / zero-second / over-100-mph / over-12-hour trips (F3–F9).** None of the four questions uses miles or duration, so a bad distance or duration does not distort any figure; and most of these rows have an ordinary, small fare, so they look like real (short or cancelled) trips with bad meter data. They stay as reported trips. The *Robustness check* near the end re-runs the headline results without them.
- **`trip_total` ≠ fare + tips + tolls + extras (F1).** The gap is never negative and is a small, round amount (mostly \$0.50, sometimes \$2.50–\$3.00), mainly on `Credit Card` and `Mobile` trips. I assume it is a card or other surcharge that is not itemised. **`trip_total` is used as reported revenue**, since it is what was charged; the un-itemised total is small (see above).
- **Fare of exactly 0 (F10)** and **tip larger than fare (F11).** Left in; they have little effect on aggregate figures. Tip percentage is computed as total tips ÷ total fare, so zero fares cannot cause a division error.
- **No pickup community area (F12).** Kept: this is reported in Question 2, not removed.

**Other assumptions**
- Company names are tidied before counting (next cell). The same business appears under two spellings, which would otherwise split its trips.
- `payment_type` values are used as reported (including the small `No Charge` and `Dispute` groups). I do not interpret the `Prcard` label.
- Very small differences in "week" boundaries are ignored: a trip is assigned to the day of its rounded start time.

In [ ]:
# Company-name tidy-up: spellings of the same business are merged (assumption: they are the same company).
company_map = {
    'Taxicab Insurance Agency, LLC': 'Taxicab Insurance Agency Llc',
    'Choice Taxi Association Inc': 'Choice Taxi Association',
    'Taxi Affiliation Services Llc - Yell': 'Taxi Affiliation Services',   # truncated 'Yellow' brand of the same group
}
before = df.company.nunique()
df['company_clean'] = df.company.replace(company_map)
print(f"Distinct company names: {before} -> {df.company_clean.nunique()} after merging")
for k, v in company_map.items():
    print(f"  {k!r} ({(df.company == k).sum():,} trips) -> {v!r}")

# Derived time fields
df['date'] = df.start.dt.normalize()
df['dow'] = df.start.dt.day_of_week
df['hour'] = df.start.dt.hour
DAYS = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
df['day'] = pd.Categorical(df.dow.map(dict(enumerate(DAYS))), categories=DAYS, ordered=True)
print(f"\nAnalysis set ready: {len(df):,} trips, {df.date.nunique()} days")

## 3. Question 1 – When is demand highest?
Trips counted by day of week and by hour of trip start (rounded start time).

In [ ]:
by_day = df.groupby('day', observed=True).size().rename('trips').to_frame()
by_day['share of week %'] = by_day.trips / by_day.trips.sum() * 100
by_day['vs. daily average %'] = (by_day.trips / by_day.trips.mean() - 1) * 100
display(by_day.style.format({'trips': '{:,.0f}', 'share of week %': '{:.1f}', 'vs. daily average %': '{:+.1f}'}))

by_hour = df.groupby('hour').size().rename('trips').to_frame()
by_hour['share of week %'] = by_hour.trips / by_hour.trips.sum() * 100
print("Busiest hours:", by_hour.trips.nlargest(3).to_dict())
print("Quietest hours:", by_hour.trips.nsmallest(3).to_dict())
heat = df.groupby(['day', 'hour'], observed=True).size().unstack(fill_value=0).reindex(columns=range(24), fill_value=0)
print("Busiest day-hour cell:", heat.stack().idxmax(), int(heat.stack().max()), "| quietest:", heat.stack().idxmin(), int(heat.stack().min()))

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.2))
cols = [BLUE if d in DAYS[:5] else ORANGE for d in DAYS]
bars = ax.bar(DAYS, by_day.trips, color=cols, width=0.62)
for b, v in zip(bars, by_day.trips):
    ax.text(b.get_x() + b.get_width()/2, v + 250, f"{v:,.0f}", ha='center', va='bottom', fontsize=9.5, color=INK2)
ax.set_title('Reported taxi trips by day of week, 11–17 Sep 2023')
ax.set_xlabel('Day of trip start'); ax.set_ylabel('Trips')
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.set_ylim(0, by_day.trips.max() * 1.12); ax.grid(axis='x', visible=False)
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=BLUE, label='Weekday (Mon–Fri)'), Patch(color=ORANGE, label='Weekend (Sat–Sun)')], frameon=False, loc='lower left')
plt.tight_layout(); plt.show()

In [ ]:
from matplotlib.colors import LinearSegmentedColormap
seq = LinearSegmentedColormap.from_list('blue_seq', ['#cde2fb', '#86b6ef', '#3987e5', '#256abf', '#184f95', '#0d366b'])
fig, ax = plt.subplots(figsize=(11, 3.8))
im = ax.imshow(heat.values, aspect='auto', cmap=seq, vmin=0)
ax.set_xticks(range(24)); ax.set_xticklabels([f'{h:02d}' for h in range(24)])
ax.set_yticks(range(7)); ax.set_yticklabels(DAYS)
ax.set_xticks(np.arange(-.5, 24, 1), minor=True); ax.set_yticks(np.arange(-.5, 7, 1), minor=True)
ax.grid(which='minor', color=SURFACE, linewidth=2); ax.grid(which='major', visible=False)
ax.tick_params(which='minor', length=0)
for s in ax.spines.values(): s.set_visible(False)
ax.set_title('Trips by day and hour of start (darker = more trips)')
ax.set_xlabel('Hour of trip start (00 = midnight to 00:59, rounded start time)'); ax.set_ylabel('Day')
cb = fig.colorbar(im, ax=ax, pad=0.01); cb.set_label('Trips in that hour'); cb.outline.set_visible(False)
plt.tight_layout(); plt.show()

In [ ]:
# Average trips per hour: weekdays vs weekends. Friday shown separately because it is clearly different from Mon-Thu.
grp = {'Mon–Thu (avg of 4 days)': [0, 1, 2, 3], 'Fri': [4], 'Sat–Sun (avg of 2 days)': [5, 6]}
prof = pd.DataFrame({k: heat.iloc[v].mean() for k, v in grp.items()})
fig, ax = plt.subplots(figsize=(10, 4.4))
styles = {'Mon–Thu (avg of 4 days)': (BLUE, '-'), 'Fri': (BLUE, (0, (1, 1.6))), 'Sat–Sun (avg of 2 days)': (ORANGE, '-')}
for k, (c, ls) in styles.items():
    ax.plot(prof.index, prof[k], color=c, lw=2, ls=ls, label=k)
    ax.plot(prof.index[-1], prof[k].iloc[-1], 'o', color=c, ms=7, mec=SURFACE, mew=2)
ax.set_title('Average trips per hour of the day: weekdays vs weekend')
ax.set_xlabel('Hour of trip start'); ax.set_ylabel('Average trips in that hour, per day')
ax.set_xticks(range(0, 24, 2)); ax.set_xticklabels([f'{h:02d}' for h in range(0, 24, 2)])
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.set_xlim(-0.3, 23.5); ax.set_ylim(0); ax.legend(frameon=False, loc='upper left')
plt.tight_layout(); plt.show()

wk = df.assign(kind=np.where(df.dow < 5, 'Weekday (Mon–Fri)', 'Weekend (Sat–Sun)')).groupby('kind').agg(trips=('trip_id', 'size'), days=('date', 'nunique'))
wk['trips per day'] = wk.trips / wk.days
display(wk.style.format({'trips': '{:,.0f}', 'trips per day': '{:,.0f}'}))
print("Weekend day vs Mon-Thu day: %.0f%% fewer trips" % ((1 - wk.loc['Weekend (Sat–Sun)', 'trips per day'] / by_day.trips.iloc[:4].mean()) * 100))
late = (heat.loc[:, 0:4].sum(axis=1) / heat.sum(axis=1) * 100).round(1)
print("Share of each day's trips starting 00:00-04:59 (%):", late.to_dict())
print("Peak hour for each day:", heat.idxmax(axis=1).to_dict())

**Takeaway (Q1).** Demand is steady from Monday to Thursday (21,962–22,622 trips a day), a bit lower on Friday (20,151), and about 42% lower per day on Saturday (12,643, the quietest day) and Sunday (13,317). Wednesday is nominally the busiest day, but Mon–Thu are within 3% of each other, so I would not call any one of them busiest from a single week. Within the day, the busiest hours are 14:00–17:59, with 17:00 on top (9,468 trips), and the quietest is 03:00 (627 trips, only 45 of them on Wednesday). Weekdays have a sharp morning ramp from 06:00 to 09:00; weekends build slowly, peak in mid-afternoon, and are relatively busier between midnight and 05:00 (about 8.5% of weekend trips against 2.5–5% on weekdays, so the overnight trough is shallower, though still tiny in absolute terms: roughly 130 trips an hour at 03:00 on a weekend day). Friday follows the weekday shape at a lower volume through the day, most clearly in the morning.

## 4. Question 2 – Where do trips start?
Top 10 pickup community areas by trip count and by total `trip_total`. Areas are shown by their numeric code.

In [ ]:
area = df.groupby('pickup_community_area').agg(trips=('trip_id', 'size'), revenue=('trip_total', 'sum'))
area['share of trips %'] = area.trips / len(df) * 100
area['avg trip_total'] = area.revenue / area.trips
area.index = area.index.astype(int).rename('community area')
top_n = area.sort_values('trips', ascending=False).head(10)
top_r = area.sort_values('revenue', ascending=False).head(10)
fmt = {'trips': '{:,.0f}', 'revenue': '${:,.0f}', 'share of trips %': '{:.1f}', 'avg trip_total': '${:,.2f}'}
print("Top 10 by trip count"); display(top_n.style.format(fmt))
print("Top 10 by total trip_total"); display(top_r.style.format(fmt))

no_area = df.pickup_community_area.isna()
print(f"Trips with no pickup community area: {no_area.sum():,} = {no_area.mean():.2%} of {len(df):,} trips (raw file: {raw.pickup_community_area.isna().mean():.2%})")
print(f"  ...and their revenue: ${df.loc[no_area,'trip_total'].sum():,.0f} = {df.loc[no_area,'trip_total'].sum()/df.trip_total.sum():.2%} of total; avg trip_total ${df.loc[no_area,'trip_total'].mean():,.2f} vs ${df.loc[~no_area,'trip_total'].mean():,.2f} for trips with an area")
print(f"  ...of these, {df.loc[no_area,'pickup_centroid_latitude'].notna().sum()} still have a pickup centroid, so 'no area' is not purely 'outside Chicago'")
print(f"Top 10 areas by count hold {top_n['share of trips %'].sum():.1f}% of all trips; the top 3 hold {top_n['share of trips %'].iloc[:3].sum():.1f}%")
print("Number of distinct pickup areas:", area.shape[0])
print("Top-10 lists differ by:", sorted(set(top_n.index) ^ set(top_r.index)))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.8))
for ax, t, col, ttl, xl, f in [
    (axes[0], top_n, 'trips', 'Top 10 pickup areas by trips', 'Trips', lambda v: f'{v:,.0f}'),
    (axes[1], top_r.sort_values('revenue', ascending=False), 'revenue', 'Top 10 pickup areas by total trip_total', 'Total trip_total ($)', lambda v: f'${v/1000:,.0f}k')]:
    t = t.iloc[::-1]
    bars = ax.barh([str(i) for i in t.index], t[col], color=BLUE, height=0.62)
    for b, v in zip(bars, t[col]):
        ax.text(v + t[col].max()*0.012, b.get_y() + b.get_height()/2, f(v), va='center', fontsize=9, color=INK2)
    ax.set_title(ttl, fontsize=12); ax.set_xlabel(xl); ax.set_ylabel('Pickup community area (code)')
    ax.set_xlim(0, t[col].max() * 1.16); ax.grid(axis='y', visible=False)
    ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _, c=col: f'{x:,.0f}' if c == 'trips' else f'${x/1000:,.0f}k'))
    ax.xaxis.set_major_locator(mtick.MultipleLocator(10000 if col == 'trips' else 500000))
    ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _, c=col: f'{x:,.0f}' if c == 'trips' else f'${x/1e6:,.1f}M' if x else '$0'))
fig.suptitle('Where reported taxi trips start (community areas with a recorded pickup area)', x=0.01, ha='left', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

**Takeaway (Q2).** Three community areas (76, 8 and 32) account for 60.5% of all reported trips, and the top 10 account for 85.1%. The top 10 by trip count and by total `trip_total` are the same ten areas, in a slightly different order (area 56 ranks 4th by revenue but 6th by trips). Area 76 is unusual: 23.3% of trips but 43% of revenue, with an average `trip_total` of \$54.74; area 56 is similar (\$46.55), versus \$16–26 elsewhere in the top 10. That pattern points to longer trips from those two areas, but explaining why would need outside data on what the areas are. 3,747 trips (2.77% of the analysis set) have no pickup community area; these are mostly trips beginning outside Chicago (per `dataset.md`), and they average a higher `trip_total` (\$34.46 vs \$29.59), as longer out-of-city trips would. 38 of them still have a pickup centroid, so 'no area' is not purely 'outside Chicago'.

## 5. Question 3 – How do riders pay and tip?
Per `payment_type`: trips, total revenue (`trip_total`), average fare. Tip % = total tips ÷ total fare. Because cash tips are generally not recorded, tip percentages across payment types are **not like-for-like**; the meaningful comparison is between `Credit Card` and `Mobile`, where tips are recorded for >90% of trips.

In [ ]:
g = df.groupby('payment_type')
pay = pd.DataFrame({
    'trips': g.size(),
    'share of trips %': g.size() / len(df) * 100,
    'total revenue': g.trip_total.sum(),
    'share of revenue %': g.trip_total.sum() / df.trip_total.sum() * 100,
    'avg fare': g.fare.mean(),
    'total fare': g.fare.sum(),
    'total tips': g.tips.sum(),
})
pay['tips % of fare'] = pay['total tips'] / pay['total fare'] * 100
pay['trips with a tip %'] = g.apply(lambda s: (s.tips > 0).mean() * 100)
pay['trips with a tip'] = g.apply(lambda s: (s.tips > 0).sum())
pay = pay.sort_values('trips', ascending=False)
f3 = {'trips': '{:,.0f}', 'share of trips %': '{:.1f}', 'total revenue': '${:,.0f}', 'share of revenue %': '{:.1f}', 'avg fare': '${:,.2f}',
      'total fare': '${:,.0f}', 'total tips': '${:,.0f}', 'tips % of fare': '{:.1f}', 'trips with a tip %': '{:.1f}',
      'trips with a tip': '{:,.0f}'}
tot = pd.DataFrame({'trips': [len(df)], 'total revenue': [df.trip_total.sum()], 'avg fare': [df.fare.mean()]}, index=['ALL'])
display(pay[['trips', 'share of trips %', 'total revenue', 'share of revenue %', 'avg fare']].style.format(f3))
display(tot.style.format(f3))
print("Tipping (tips recorded reliably only for Credit Card and Mobile - read the others with care)")
display(pay[['total fare', 'total tips', 'tips % of fare', 'trips with a tip %', 'trips with a tip']].style.format(f3))
card = pay.loc[['Credit Card', 'Mobile']]
print("Credit Card + Mobile combined tips %% of fare: %.1f%%" % (card['total tips'].sum() / card['total fare'].sum() * 100))
print("Tips as share of trip_total, all trips: %.1f%%" % (df.tips.sum() / df.trip_total.sum() * 100))

In [ ]:
order = pay.index.tolist()
reliable = {'Credit Card', 'Mobile'}
fig, axes = plt.subplots(1, 2, figsize=(11, 4.6))
ax = axes[0]
r = pay.loc[order[::-1]]
bars = ax.barh(r.index, r['trips'], color=BLUE, height=0.62)
for b, v, sh in zip(bars, r['trips'], r['share of trips %']):
    ax.text(v + r['trips'].max()*0.012, b.get_y()+b.get_height()/2, f'{v:,.0f}  ({sh:.0f}%)', va='center', fontsize=9, color=INK2)
ax.set_title('Trips by payment type', fontsize=12); ax.set_xlabel('Trips'); ax.set_ylabel('Payment type')
ax.set_xlim(0, r['trips'].max()*1.3); ax.grid(axis='y', visible=False)
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax = axes[1]
r2 = r
cols = [BLUE if i in reliable else GRAY for i in r2.index]
bars = ax.barh(r2.index, r2['tips % of fare'], color=cols, height=0.62)
for b, v in zip(bars, r2['tips % of fare']):
    ax.text(v + 0.3, b.get_y()+b.get_height()/2, f'{v:.1f}%', va='center', fontsize=9, color=INK2)
ax.set_title('Tips as % of fare', fontsize=12); ax.set_xlabel('Total tips ÷ total fare (%)'); ax.set_ylabel('')
ax.set_xlim(0, max(r2['tips % of fare'].max()*1.2, 20)); ax.grid(axis='y', visible=False)
ax.legend(handles=[Patch(color=BLUE, label='Tips recorded for >90% of trips'), Patch(color=GRAY, label='Tips mostly not recorded: not comparable')],
          frameon=False, loc='lower right', fontsize=9)
fig.suptitle('How riders pay and tip, 11–17 Sep 2023 (reported trips)', x=0.01, ha='left', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

**Takeaway (Q3).** Credit cards dominate: 44.5% of trips and 59.6% of revenue (\$2.39M), with the highest average fare among the main types (\$28.72). Cash is the second most common method (29.3% of trips) but earns a smaller share of revenue (19.6%) because its average fare is lower (\$18.14); `Mobile` has the lowest average fare of the main types (\$17.81). Tips can only be compared between `Credit Card` (22.2% of fare) and `Mobile` (20.5%), since only these two record tips for over 90% of trips; the two are close, with card slightly higher. Cash, `Prcard` and `Unknown` show almost no tips (0.0–0.7%), which almost certainly means tips are not captured for those payment types rather than that those riders do not tip. The overall tip share of `trip_total` (11.3%) is therefore a floor, not a typical tip.

## 6. Question 4 – Which companies carry the trips?
Top 10 companies by trip count after merging duplicate spellings (see cleaning). "Distinct taxis" counts unique `taxi_id` values per company within the week.

In [ ]:
c = df.groupby('company_clean').agg(trips=('trip_id', 'size'), revenue=('trip_total', 'sum'), taxis=('taxi_id', 'nunique'))
c['share of all trips %'] = c.trips / len(df) * 100
c['share of revenue %'] = c.revenue / df.trip_total.sum() * 100
c['trips per taxi'] = c.trips / c.taxis
c['avg trip_total'] = c.revenue / c.trips
c = c.sort_values('trips', ascending=False)
top_c = c.head(10).copy(); top_c.index.name = 'company'
f4 = {'trips': '{:,.0f}', 'revenue': '${:,.0f}', 'taxis': '{:,.0f}', 'share of all trips %': '{:.1f}', 'share of revenue %': '{:.1f}', 'trips per taxi': '{:.1f}', 'avg trip_total': '${:,.2f}'}
display(top_c[['trips', 'revenue', 'share of all trips %', 'taxis', 'share of revenue %', 'trips per taxi', 'avg trip_total']].style.format(f4))
print(f"Top 10 together: {top_c.trips.sum():,} trips = {top_c.trips.sum()/len(df):.1%} of all trips; {top_c.revenue.sum()/df.trip_total.sum():.1%} of revenue")
print(f"Top 3 together: {top_c.trips.iloc[:3].sum()/len(df):.1%} of trips. Companies in the data: {len(c)}")
print(f"Distinct taxis overall: {df.taxi_id.nunique():,}; taxis seen under more than one company: {(df.groupby('taxi_id').company_clean.nunique() > 1).sum()} (so company taxi counts can sum to slightly more than the total)")
print("Rank without name merging (for reference):", df.company.value_counts().head(5).to_dict())

In [ ]:
t = top_c.iloc[::-1]
fig, ax = plt.subplots(figsize=(9.5, 4.8))
bars = ax.barh(t.index, t.trips, color=BLUE, height=0.62)
for b, v, s, k in zip(bars, t.trips, t['share of all trips %'], t.taxis):
    ax.text(v + t.trips.max()*0.012, b.get_y()+b.get_height()/2, f'{v:,.0f}  ({s:.1f}% of trips, {k:,} taxis)', va='center', fontsize=9, color=INK2)
ax.set_title('Top 10 taxi companies by reported trips, 11–17 Sep 2023')
ax.set_xlabel('Trips'); ax.set_ylabel('Company'); ax.set_xlim(0, t.trips.max()*1.5); ax.grid(axis='y', visible=False)
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.tight_layout(); plt.show()

**Takeaway (Q4).** Flash Cab and Taxi Affiliation Services each carry about 19% of trips (25,896 and 25,748) and are in effect tied; together with Taxicab Insurance Agency, Sun Taxi and City Service they carry 71% of trips, and the top 10 carry 92%. Revenue shares track trip shares closely (top 10: 92.1% of revenue). Flash Cab gets the most trips per taxi (64.6) and Taxi Affiliation Services has the largest fleet that was active in the week (459 taxis). Average `trip_total` per trip varies from \$26 (Flash Cab, Blue Ribbon) to \$34 (5 Star Taxi), so a company's revenue rank can differ from its trip rank. Merging the duplicate company spellings matters here: without it Taxicab Insurance Agency would rank 4th (13,800 trips) behind Sun Taxi.

## 7. Robustness check
Do the headline results change if the "kept but flagged" rows with unreliable miles/duration (F5 zero miles *and* zero seconds, F6, F7, F8), zero fares (F10) and tips larger than the fare (F11) are also removed?

In [ ]:
strict_mask = ~(flags['F5  Zero miles AND zero seconds'] | flags['F6  Miles > 0 but zero seconds'] | flags['F7  Speed above 100 mph'] |
                flags['F8  Duration over 12 hours'] | flags['F10 Fare is exactly 0'] | flags['F11 Tip larger than the fare'])
strict = df[strict_mask.loc[df.index]]
def headline(d):
    d = d.copy()
    hh = d.groupby('hour').size()
    cc = d.groupby('company_clean').size().sort_values(ascending=False)
    ar = d.groupby('pickup_community_area').size().sort_values(ascending=False)
    ar_r = d.groupby('pickup_community_area').trip_total.sum().sort_values(ascending=False)
    cm = d[d.payment_type.isin(['Credit Card', 'Mobile'])]
    return {'trips': len(d),
            'busiest day': d.groupby('day', observed=True).size().idxmax(), 'quietest day': d.groupby('day', observed=True).size().idxmin(),
            'busiest hour': hh.idxmax(), 'quietest hour': hh.idxmin(),
            'top-5 areas by trips': list(ar.index[:5].astype(int)), 'top-10 areas by trips (set equal?)': set(ar.index[:10]),
            'top-10 areas by revenue (set)': set(ar_r.index[:10]),
            'no pickup area %': round(d.pickup_community_area.isna().mean() * 100, 2),
            'card+mobile tips % of fare': round(cm.tips.sum() / cm.fare.sum() * 100, 1),
            'top-10 companies (list)': list(cc.index[:10])}
a, b = headline(df), headline(strict)
cmp = pd.DataFrame({'analysis set': a, 'stricter set': b})
cmp['same?'] = [('—' if k == 'trips' else (f'diff {b[k]-a[k]:+.2f}' if isinstance(a[k], float) else str(a[k] == b[k]))) for k in a]
print(f"Stricter set removes a further {len(df)-len(strict):,} trips ({(len(df)-len(strict))/len(df):.1%})")
display(cmp.astype(str))

## 8. Limitations and a possible follow-up
- **One week only.** Each weekday appears once, so day-to-day differences (for example Friday being lower than Mon–Thu) could be one-off; this week cannot show what is "typical". It was an ordinary week, but the brief excludes comparisons with other weeks.
- **Reported trips only.** Counts, revenue and company shares are of trips reported to the City, not the whole market. Companies that report less completely look smaller.
- **Timestamps are rounded** to 15 minutes, so hour-level patterns are accurate to about ±8 minutes.
- **Tips** are only meaningful for card and mobile payments; tips on cash trips are generally not captured, so overall tip % understates real tipping.
- **Possible follow-up:** areas 76 and 56 earn roughly two to three times the average `trip_total` of the other top-10 areas, so a look at what drives those trips (distance, time of day, payment type) once the areas are identified would explain much of the revenue picture.